# MIMO-04 : Capstone — digérer la détection MIMO par flips

**Navigation** : [Index](README.md) | [<< MIMO-03 (Converse-Native)](MIMO-03-Converse-Native-Lean.ipynb) | [Lean-22 (Galois) >>](../GALOIS/GALOIS-01-Probleme-Inverse-M23-Python.ipynb)

Ce carnet **digère** la sous-série MIMO plutôt que d'en ajouter une marche : il
reprend le théorème de tête, la borne de queue de norme et le seuil `2·log N`
que les trois carnets prouvent séparément, et il les met **à l'épreuve sur des
instances calculées**. C'est le pendant, pour MIMO, du capstone
[Lean-20](../Lean-20-Capstone-Digestions-Tao-Python.ipynb) de la série
principale : le même geste de digestion, un autre objet.

Il est **exécutable seul** : noyau Python, aucune dépendance au lake, aucune
commande Lean.


## 1. Le geste : digérer, pas seulement vérifier

Les trois marches de la sous-série remplissent trois fonctions distinctes :
**lire** un théorème et ses hypothèses (MIMO-01), **simuler** le détecteur réel
et sa borne (MIMO-02), **exécuter** le lake dans un noyau Lean (MIMO-03). Aucune
ne fait le geste de synthèse : prendre un énoncé formel, en instancier les
hypothèses sur un objet calculable, et **constater** si la conclusion tient.

C'est ce que ce carnet fait, sur deux énoncés qui portent tout le résultat :

| Énoncé formel | Où il vit | Ce qu'on en fait ici |
|---|---|---|
| `descent_target_before_ceiling` | `Descent.lean` (Prop. 9.1) | on instancie la descente à flips sur une instance MIMO et on **mesure** les quatre faits de sa preuve |
| `norm_concentration` / `column_norm_tail` | `NormTails.lean` | on **confronte** la queue `2·exp(−t²/2)` à un tirage Monte-Carlo, puis on en dérive le seuil `2·log N` par union bound |

Le carnet ne redémontre rien : il **relie** un énoncé à un nombre. Là où
l'expérience ne peut pas décider, il le dit.


## 2. Où entrer : les trois marches

| Marche | Carnet | Geste | Prérequis |
|---|---|---|---|
| base | [MIMO-01 — Descente-Budget](MIMO-01-Descente-Budget-Lean-Python.ipynb) | lire le squelette combinatoire (`Descent.lean`, **sans Mathlib**) | aucun Lean — le carnet s'ouvre sur du texte |
| milieu | [MIMO-02 — Detection-Flips](MIMO-02-Detection-Flips-Lean-Python.ipynb) | simuler le détecteur, confronter la borne `NormTails` | noyau Python, disposition à lire un énoncé |
| sommet | [MIMO-03 — Converse-Native](MIMO-03-Converse-Native-Lean.ipynb) | exécuter le lake entier, énoncé par énoncé | WSL + Lean 4 + le lake |

La progression est **monotone en prérequis** : c'est le contrat d'un escalier.
Ce carnet se place **après** les trois, et n'exige que le noyau Python.


## 3. La marche faite à la main : la descente à flips

`Descent.lean` prouve la Proposition 9.1 sur un espace d'états **abstrait** :
`accept : σ → σ → Prop`, `cost : σ → ℕ`, et quatre faits enchaînés —
la décroissance se propage à toute la queue du run, un run ne se répète jamais,
le nombre de flips est majoré par le coût de départ, et sous une barrière `B` il
est majoré par `B`.

On instancie `σ` par les `2^N` vecteurs de signes d'un petit canal MIMO et
`cost` par le **rang** de l'objectif `‖y − Hx‖²`. Le rang est un entier : la
décroissance stricte de l'objectif devient une décroissance stricte d'au moins
une unité, ce qui est **exactement** l'hypothèse `cost : σ → ℕ` du module. Un
coût réel ne la fournirait pas — on y revient en section 4.

Le flip `x_i → −x_i` a pour variation de coût `4·x_i·(rᵀh_i) + 4·‖h_i‖²` avec
`r = y − Hx` : le flip est accepté si cette quantité est **strictement**
négative.


In [1]:
# La descente a flips, telle que `Descent.lean` la formalise : on ne suppose
# rien, on mesure. Instance MIMO jouet, entierement deterministe (graine fixe).
import numpy as np
from itertools import product

rng = np.random.default_rng(20261011)
N, M, SIGMA = 10, 14, 0.6

H = rng.normal(size=(M, N))                        # canal, entrees i.i.d. N(0,1)
x_vrai = rng.choice(np.array([-1.0, 1.0]), size=N) # signal emis (BPSK)
bruit = SIGMA * rng.normal(size=M)
y = H @ x_vrai + bruit


def cout(x):
    """Objectif du detecteur : ||y - Hx||^2."""
    r = y - H @ x
    return float(r @ r)


def gains_de_flip(x):
    """Variation de cout de chaque flip x_i -> -x_i.

    r' = r + 2*x_i*h_i, d'ou cout(x^i) - cout(x) = 4*x_i*(r . h_i) + 4*||h_i||^2.
    """
    r = y - H @ x
    return 4.0 * (x * (H.T @ r) + np.sum(H * H, axis=0))


# Le rang de l'objectif : un cout ENTIER, strictement monotone en l'objectif.
etats = np.array(list(product([-1.0, 1.0], repeat=N)))        # les 2^N etats
objectifs = np.array([cout(x) for x in etats])
ordre = np.argsort(objectifs, kind="stable")
rang = np.empty(len(etats), dtype=int)
rang[ordre] = np.arange(len(etats))
index = {tuple(s): k for k, s in enumerate(etats)}


def cout_entier(x):
    """cost : sigma -> N, au sens de `Descent.lean`."""
    return int(rang[index[tuple(x)]])


# La descente : a chaque pas, le meilleur flip s'il DECROIT strictement.
x0 = np.ones(N)                                    # aucun signe exploite au depart
run = [x0.copy()]
while True:
    delta = gains_de_flip(run[-1])
    i = int(np.argmin(delta))
    if delta[i] >= 0.0:
        break                                      # etat terminal : hterm
    suivant = run[-1].copy()
    suivant[i] = -suivant[i]
    run.append(suivant)

cout_reel = [cout(x) for x in run]
cout_nat = [cout_entier(x) for x in run]

print("canal %dx%d, bruit sigma = %.2f, signal emis recupere : %s"
      % (M, N, SIGMA, np.array_equal(run[-1], x_vrai)))
print()
print("pas |  cout reel  | cout entier")
for k, (cr, cn) in enumerate(zip(cout_reel, cout_nat)):
    print("%3d | %11.6f | %5d" % (k, cr, cn))
print()
print("flips acceptes : %d" % (len(run) - 1))
print("aucun flip n'ameliore l'etat final : %s"
      % bool(np.all(gains_de_flip(run[-1]) >= 0.0)))

canal 14x10, bruit sigma = 0.60, signal emis recupere : True

pas |  cout reel  | cout entier
  0 |  308.745023 |   721
  1 |  145.193729 |   127
  2 |   74.455617 |    16
  3 |   45.950429 |     4
  4 |    2.667405 |     0

flips acceptes : 4
aucun flip n'ameliore l'etat final : True


### Lecture du résultat

La trajectoire descend de `cost = 721` à `cost = 0` en **4 flips acceptés**. Le
coût entier décroît strictement à chaque pas, ce qui est le contenu de
`run_tail_cost_lt` : le coût de tout état visité après le départ est strictement
inférieur à celui du départ.

L'état terminal n'offre plus aucun flip améliorant — c'est l'hypothèse `hterm`
du théorème. Le théorème ne dit **pas** qu'un tel état est l'optimum global : il
dit que le run y arrive en moins de `M_N` flips. Ici il se trouve que le rang
final vaut `0`, donc que la descente a atteint l'optimum global — c'est une
observation sur cette instance, pas une conséquence de la Proposition 9.1.


In [2]:
# Les quatre faits de `Descent.lean`, verifies sur le run qui vient d'etre
# calcule. Chacun est un theoreme du module ; on les confronte un par un.
distincts = len(set(index[tuple(x)] for x in run))
B = cout_nat[0]        # barriere de confinement : le cout ne remonte jamais
M_N = B + 1            # plafond de flips, strictement au-dessus de la barriere
n_flips = len(run) - 1

faits = [
    ("run_tail_cost_lt               (decroissance stricte)",
     all(b < a for a, b in zip(cout_nat, cout_nat[1:]))),
    ("run_nodup                      (aucune repetition)",
     distincts == len(run)),
    ("run_length_le_cost             (flips <= cout de depart)",
     n_flips <= B),
    ("descent_flips_le_barrier       (flips <= B)",
     n_flips <= B),
    ("descent_target_before_ceiling  (cible atteinte, flips < M_N)",
     bool(np.all(gains_de_flip(run[-1]) >= 0.0)) and n_flips < M_N),
]

for libelle, verdict in faits:
    print("%-52s %s" % (libelle, "OK" if verdict else "ECHEC"))
print()
print("B = %d, M_N = %d, flips = %d, etats distincts = %d sur %d pas"
      % (B, M_N, n_flips, distincts, len(run)))

run_tail_cost_lt               (decroissance stricte) OK
run_nodup                      (aucune repetition)   OK
run_length_le_cost             (flips <= cout de depart) OK
descent_flips_le_barrier       (flips <= B)          OK
descent_target_before_ceiling  (cible atteinte, flips < M_N) OK

B = 721, M_N = 722, flips = 4, etats distincts = 5 sur 5 pas


### Lecture du résultat

Les cinq lignes sont les cinq énoncés du module, sans exception. Le fait le plus
intéressant est `run_length_le_cost` : **`4 ≤ 721`**, une borne vraie mais
grossière. Elle est grossière *par construction* — la preuve n'utilise que le
fait que le coût est un entier qui décroît d'au moins `1` à chaque flip, jamais
la géométrie du problème. C'est la barrière `B` de la Phase 2 (`Objective.lean`)
qui resserre la borne ; ici on a pris `B = cost(s₀)`, le cas dégénéré où la
barrière ne contraint rien.


## 4. Le défaut comme dissociation : le témoin d'échec

La suite du module ne fait pas que prouver : elle **donne un nom au défaut**.
`DescentFailure` est un témoin — une paire d'états consécutifs dont le flip est
accepté sans que le coût décroisse strictement — et
`failure_witness_denies_global_strict` dit que ce témoin **réfute** l'hypothèse
globale. Le commentaire du module est explicite : un échec de décroissance n'est
pas un résidu numérique à écarter, c'est une dissociation à enregistrer.

On construit ici le témoin **exactement**, sans flottement : un canal identité
sans bruit rend l'objectif `‖y − Hx‖² = ‖x‖² = M` **constant**. Tous les gains
de flip valent alors `0` — l'égalité, pas l'approximation.


In [3]:
# Temoin exact : objectif constant, donc variation de flip nulle partout.
H_sym = np.eye(M)
y_sym = np.zeros(M)


def gains_sym(x):
    r = y_sym - H_sym @ x
    return 4.0 * (x * (H_sym.T @ r) + np.sum(H_sym * H_sym, axis=0))


x_sym = np.ones(M)
cout_const = float((y_sym - H_sym @ x_sym) @ (y_sym - H_sym @ x_sym))
print("objectif de tout etat : %.1f   (constant)" % cout_const)
print("gains de flip possibles :", np.unique(gains_sym(x_sym)))
print("regle STRICTE  (accepte si gain < 0)  : %d flip accepte"
      % int(np.sum(gains_sym(x_sym) < 0.0)))
print("regle RELACHEE (accepte si gain <= 0) : %d flips acceptes"
      % int(np.sum(gains_sym(x_sym) <= 0.0)))

# Run sous la regle relachee : on accepte le premier flip non strict.
x_rel = x_sym.copy()
visites = []
for _ in range(6):
    visites.append(tuple(x_rel))
    delta = gains_sym(x_rel)
    i = int(np.argmin(delta))
    x_rel[i] = -x_rel[i]
print()
print("run relache : %d pas, %d etats distincts, %d repetitions"
      % (len(visites), len(set(visites)), len(visites) - len(set(visites))))
print("cycle de periode 2 : l'etat de depart est retrouve au 2e pas : %s"
      % (visites[0] == visites[2]))

# Le temoin lui-meme : la paire (s, u) du constructeur `DescentFailure.here`.
s, u = np.ones(M), np.ones(M)
u[0] = -1.0
accepte = bool(np.all(gains_sym(s) <= 0.0))
decroit = bool(float((y_sym - H_sym @ u) @ (y_sym - H_sym @ u))
               < float((y_sym - H_sym @ s) @ (y_sym - H_sym @ s)))
print()
print("temoin : accept(s, u) = %s et cost(u) < cost(s) = %s  -> DescentFailure.here"
      % (accepte, decroit))
print("donc 'pour tout flip accepte, le cout decroit strictement' est FAUX : %s"
      % (accepte and not decroit))

objectif de tout etat : 14.0   (constant)
gains de flip possibles : [0.]
regle STRICTE  (accepte si gain < 0)  : 0 flip accepte
regle RELACHEE (accepte si gain <= 0) : 14 flips acceptes

run relache : 6 pas, 2 etats distincts, 4 repetitions
cycle de periode 2 : l'etat de depart est retrouve au 2e pas : True

temoin : accept(s, u) = True et cost(u) < cost(s) = False  -> DescentFailure.here
donc 'pour tout flip accepte, le cout decroit strictement' est FAUX : True


### Lecture du résultat

Le témoin est exhibé, et il fait exactement ce que le module annonce : il
**paie** l'hypothèse de stricte décroissance. Sous la règle relâchée, le run
revient sur son état de départ en deux pas — `run_nodup` tombe, et avec lui
`run_length_le_cost` : sans décroissance stricte, plus rien ne borne le nombre
de flips, et l'algorithme peut boucler indéfiniment.

C'est la valeur pédagogique du témoin : au lieu de dire « l'hypothèse est
nécessaire », on **produit l'objet** qui la rend nécessaire. Le fichier ne
prouve pas seulement `descent_target_before_ceiling` sous `hstrict`, il montre
ce qui arrive quand `hstrict` tombe.


## 5. La borne de queue et le seuil `2·log N`

`NormTails.lean` ne prouve pas directement le seuil : il prouve une **queue de
norme**. Le bruit `w` et chaque colonne `hᵢ` du canal sont des vecteurs
gaussiens standards de dimension `M`, la norme euclidienne est `1`-Lipschitz, et
le théorème de concentration du lake externe SLT donne

    P(|‖X‖ − E‖X‖| ≥ t)  ≤  2·exp(−t²/2).

Le seuil `2·log N` vient **après**, d'un union bound sur les `N` colonnes : la
probabilité qu'*une* colonne sorte de l'intervalle est majorée par
`N·2·exp(−t²/2)`, et demander que ce majorant passe sous `ε` donne

    t ≥ √(2·log(2N/ε)).

On vérifie la borne d'abord, on en dérive le seuil ensuite.


In [4]:
# La queue de norme, confrontee a l'experience. K tirages par dimension,
# plusieurs seuils t, et la borne `2*exp(-t^2/2)` du module.
K = 20000
gen = np.random.default_rng(7)
print("M   | E||X||  sqrt(M) |  t   | mesure   | borne 2exp(-t2/2) | verdict")
print("-" * 74)
violations = 0
for m in (4, 16, 64, 256):
    X = gen.normal(size=(K, m))
    norme = np.linalg.norm(X, axis=1)
    moyenne = norme.mean()
    for t in (1.0, 2.0, 3.0):
        mesure = float(np.mean(np.abs(norme - moyenne) >= t))
        borne = 2.0 * np.exp(-t * t / 2.0)
        ok = mesure <= borne
        violations += 0 if ok else 1
        print("%3d | %7.3f %7.3f | %4.1f | %8.5f | %17.5f | %s"
              % (m, moyenne, np.sqrt(m), t, mesure, borne, "OK" if ok else "VIOLATION"))
print()
print("violations de norm_concentration / noise_norm_tail / column_norm_tail : %d"
      % violations)

M   | E||X||  sqrt(M) |  t   | mesure   | borne 2exp(-t2/2) | verdict
--------------------------------------------------------------------------
  4 |   1.879   2.000 |  1.0 |  0.14295 |           1.21306 | OK
  4 |   1.879   2.000 |  2.0 |  0.00360 |           0.27067 | OK
  4 |   1.879   2.000 |  3.0 |  0.00005 |           0.02222 | OK
 16 |   3.931   4.000 |  1.0 |  0.15480 |           1.21306 | OK
 16 |   3.931   4.000 |  2.0 |  0.00450 |           0.27067 | OK
 16 |   3.931   4.000 |  3.0 |  0.00005 |           0.02222 | OK
 64 |   7.967   8.000 |  1.0 |  0.15715 |           1.21306 | OK
 64 |   7.967   8.000 |  2.0 |  0.00490 |           0.27067 | OK
 64 |   7.967   8.000 |  3.0 |  0.00005 |           0.02222 | OK
256 |  15.991  16.000 |  1.0 |  0.15350 |           1.21306 | OK
256 |  15.991  16.000 |  2.0 |  0.00445 |           0.27067 | OK
256 |  15.991  16.000 |  3.0 |  0.00005 |           0.02222 | OK

violations de norm_concentration / noise_norm_tail / column_norm_tail : 0


### Du union bound au seuil

Une colonne qui sort de l'intervalle ne suffit pas à faire échouer le détecteur :
il faut qu'**aucune** des `N` colonnes n'en sorte. Le passage de « une » à
« toutes » est un union bound, et la colle qui le referme est
`one_sub_pow_le_exp_mul` — le module prouve `(1−p)^n ≤ exp(−n·p)`, ce qui
remplace un produit de `N` termes par une exponentielle. On vérifie les deux
ensemble.


In [5]:
# Du union bound sur les N colonnes au seuil `2 log N`, puis verification
# de la colle du module : `one_sub_pow_le_exp_mul`, (1-p)^n <= exp(-n p).
epsilon = 0.05
print("N     | t requis (union bound) | sqrt(2 log N) | N*2exp(-t2/2)")
print("-" * 66)
for n_col in (10, 100, 1000):
    t_req = np.sqrt(2.0 * np.log(2.0 * n_col / epsilon))
    print("%5d | %22.3f | %13.3f | %.5f"
          % (n_col, t_req, np.sqrt(2 * np.log(n_col)), n_col * 2 * np.exp(-t_req ** 2 / 2)))

print()
print("(1-p)^n <= exp(-n p) sur une grille de p et n :")
colle_ok = True
for p in (0.01, 0.1, 0.3, 0.5):
    for n_puis in (1, 5, 20, 100):
        if not (1 - p) ** n_puis <= np.exp(-n_puis * p) + 1e-15:
            colle_ok = False
print("  %s" % ("verifiee partout" if colle_ok else "CONTREDITE"))

print()
print("lecture : t croit comme sqrt(2 log N), donc LENTEMENT en N ;")
print("a N=1000 le seuil vaut %.2f contre %.2f a N=10 (x%.1f pour x%d de colonnes)."
      % (np.sqrt(2 * np.log(2 * 1000 / epsilon)),
         np.sqrt(2 * np.log(2 * 10 / epsilon)),
         np.sqrt(2 * np.log(2 * 1000 / epsilon)) / np.sqrt(2 * np.log(2 * 10 / epsilon)),
         100))

N     | t requis (union bound) | sqrt(2 log N) | N*2exp(-t2/2)
------------------------------------------------------------------
   10 |                  3.462 |         2.146 | 0.05000
  100 |                  4.073 |         3.035 | 0.05000
 1000 |                  4.604 |         3.717 | 0.05000

(1-p)^n <= exp(-n p) sur une grille de p et n :
  verifiee partout

lecture : t croit comme sqrt(2 log N), donc LENTEMENT en N ;
a N=1000 le seuil vaut 4.60 contre 3.46 a N=10 (x1.3 pour x100 de colonnes).


### Lecture du résultat

La borne tient sur les douze couples `(M, t)` testés : la queue de norme du
module est **vérifiée numériquement**, pas seulement énoncée. On lit aussi que
`E‖X‖ ≈ √M` (1.879 pour 2.000, 15.991 pour 16.000) — la constante que
`NormTails` laisse implicite dans `E‖X‖` et qui rend la borne exploitable : sans
elle, la queue porterait sur une quantité inconnue.

Le seuil croit comme `√(2·log N)`, donc **très lentement** : multiplier le nombre
de colonnes par 100 (de 10 à 1000) ne fait passer le seuil que de 3.46 à 4.60.
C'est ce qui rend le détecteur utilisable à grande dimension, et c'est le
`2·log N` du titre de la sous-série.

La colle `one_sub_pow_le_exp_mul` est vérifiée à part : c'est elle qui fait
passer du produit des `N` événements indépendants à l'exponentielle, sans quoi
le union bound ne se refermerait pas.


## 6. Mesurer la surface du lake — et relire l'instrument

Un théorème n'existe, pour un lecteur, que si un carnet le montre. La mesure de
cette exposition se fait d'ordinaire avec l'organe canonique
`scripts/lean/scan_lake_notebook_visibility.py --lake mimo_lean`. On le refait
ici **depuis les sources**, en regardant au passage ce que l'organe lui-même
compte.

Deux chiffres vont apparaître, et l'écart entre les deux est le vrai résultat de
cette section.


In [6]:
# La surface de `mimo_lean`, comptee depuis les sources, en francais
# (les jumeaux `_en.lean` portent la meme substance : convention i18n #4980).
import json
import pathlib
import re

MODIF = r"(?:private|protected|noncomputable|partial|unsafe|scoped|local)"
# Le motif de l'organe canonique : pseudo-lettre ASCII pour le nom declare.
MOTIF_ORGANE = re.compile(
    r"^\s*(?:@\[[^\]]*\]\s*)?(?:" + MODIF + r"\s+)*"
    r"(theorem|lemma|def|abbrev|instance|structure|inductive|class|opaque)\s+"
    r"([A-Za-z_][A-Za-z0-9_']*)"
)
# Meme motif, mais nom lu jusqu'au premier delimiteur (accepte les grecques).
MOTIF_LARGE = re.compile(
    r"^\s*(?:@\[[^\]]*\]\s*)?(?:" + MODIF + r"\s+)*"
    r"(theorem|lemma|def|abbrev|instance|structure|inductive|class|opaque)\s+"
    r"([^\s({\[:]+)"
)


def racine_lake():
    """Le dossier qui contient `mimo_lean`, par remontee d'ancetres."""
    d = pathlib.Path.cwd()
    for cand in [d, *d.parents]:
        if (cand / "mimo_lean").is_dir():
            return cand / "mimo_lean"
    raise FileNotFoundError("mimo_lean introuvable depuis %s" % d)


lake = racine_lake()
noms = {}
lignes_organe = 0
print("%-16s | %8s | %8s" % ("module", "organe", "large"))
print("-" * 38)
for f in sorted(lake.glob("*.lean")):
    if f.name.endswith("_en.lean"):
        continue
    texte = f.read_text(encoding="utf-8").splitlines()
    s_organe = {m.group(2) for m in (MOTIF_ORGANE.match(l) for l in texte) if m}
    s_large = {m.group(2) for m in (MOTIF_LARGE.match(l) for l in texte) if m}
    noms[f.name] = s_large
    lignes_organe += len(s_organe)
    ecart = "   <<< ECART" if len(s_organe) != len(s_large) else ""
    print("%-16s | %8d | %8d%s" % (f.name, len(s_organe), len(s_large), ecart))
total_large = sum(len(v) for v in noms.values())
print("-" * 38)
print("%-16s | %8d | %8d" % ("TOTAL", lignes_organe, total_large))

# Qui l'organe perd-il, exactement ?
for f, s in noms.items():
    perdues = {n for n in s if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_']*", n)}
    if perdues:
        print()
        print("declarations invisibles au motif de l'organe (%s) : %s"
              % (f, sorted(perdues)))
        autres = {m.group(2) for m in (MOTIF_ORGANE.match(l)
                                       for l in lake.joinpath(f).read_text(encoding="utf-8").splitlines()) if m}
        print("  l'organe les replie sur : %s"
              % sorted(n for n in autres if n.endswith("_")))

# Exposition des declarations de `Descent.lean` dans la sous-serie : citee dans
# la SOURCE d'une cellule, ou seulement lisible dans la SORTIE d'une autre ?
# Le capstone s'exclut de son propre balayage : il nomme les declarations qu'il
# mesure, donc s'il se comptait la mesure serait vraie par construction.
descent = noms["Descent.lean"]
# La sous-serie MIMO : les trois carnets `MIMO-0[123]-*`, deplaces par le
# rename de #20373 dans le dossier `MIMO/` ; le capstone `MIMO-04-*` est exclu
# du balayage (il nomme les declarations qu'il mesure).
carnets = [q for q in sorted(lake.parent.glob("MIMO/MIMO-0[123]-*.ipynb"))]

src_vu, out_vu = set(), set()
for nb_path in carnets:
    don = json.loads(nb_path.read_text(encoding="utf-8"))
    for c in don["cells"]:
        src_txt = "".join(c["source"])
        for nom in descent:
            if nom in src_txt:
                src_vu.add(nom)
        for o in c.get("outputs", []):
            blocs = o.get("text", [])
            if isinstance(blocs, str):
                blocs = [blocs]
            blocs = list(blocs) + list(o.get("data", {}).get("text/plain", []))
            for bloc in blocs:
                for nom in descent:
                    if nom in bloc:
                        out_vu.add(nom)

print()
print("exposition des %d declarations de Descent.lean dans la sous-serie :"
      % len(descent))
print("carnets balayes : %s" % ", ".join(q.stem for q in carnets))
print("(le capstone s'exclut de son propre balayage : il nomme ce qu'il mesure)")
print()
print("%-38s | %-6s | %s" % ("declaration", "source", "sortie"))
print("-" * 66)
muettes = []
for nom in sorted(descent):
    src = "oui" if nom in src_vu else "non"
    out = "oui" if nom in out_vu else "non"
    if src == "non" and out == "oui":
        muettes.append(nom)
    print("%-38s | %-6s | %s" % (nom, src, out))
print()
print("visibles seulement en sortie : %s" % (muettes or "aucune"))

module           |   organe |    large
--------------------------------------
Bridge.lean      |       15 |       15


Converse.lean    |       16 |       16
Descent.lean     |       10 |       10
lakefile.lean    |        0 |        0
Lmmse.lean       |        3 |        4   <<< ECART
NormTails.lean   |        6 |        6


Objective.lean   |        6 |        6
--------------------------------------
TOTAL            |       56 |       57

declarations invisibles au motif de l'organe (Lmmse.lean) : ['B_ρ', 'B_ρ_posSemidef']
  l'organe les replie sur : ['B_']

exposition des 10 declarations de Descent.lean dans la sous-serie :
carnets balayes : MIMO-01-Descente-Budget-Lean-Python, MIMO-02-Detection-Flips-Lean-Python, MIMO-03-Converse-Native-Lean
(le capstone s'exclut de son propre balayage : il nomme ce qu'il mesure)

declaration                            | source | sortie
------------------------------------------------------------------
DescentFailure                         | non    | oui
Run                                    | oui    | oui
descent_flips_le_barrier               | oui    | oui
descent_target_before_ceiling          | oui    | oui
failure_witness_denies_global_strict   | non    | oui
lastState                              | oui    | oui
run_length_le_cost                     | oui    |

### Lecture du résultat

L'écart est net et il est **localisé** : l'organe rend `56` là où les sources en
portent `57`. La déclaration qu'il perd est nommée — `B_ρ`, dans `Lmmse.lean` —
et la raison est dans son motif : `[A-Za-z0-9_']*` s'arrête au premier caractère
non ASCII, donc `B_ρ` se lit `B_`, exactement comme `B_ρ_posSemidef`. Les deux
se replient sur le même nom et le compte en perd un.

Rien dans le lake n'est faux : c'est **l'instrument** qui l'est, d'une seule
unité, sur les seuls noms qui portent une lettre grecque. Le chiffre `56`
recopié d'un rapport à l'autre hérite silencieusement de ce repli.

Le tableau d'exposition sépare ensuite deux choses que le comptage confond : une
déclaration peut être **citée** dans le texte d'une sortie sans jamais être
**montrée** dans une cellule source. Trois des dix déclarations de `Descent.lean`
sont dans ce cas, et ce ne sont pas trois quelconques :

- `DescentFailure`, le type du témoin d'échec ;
- `failure_witness_denies_global_strict`, qui l'exploite ;
- `run_strictdescent_excludes_failure`, le verrou que la sous-série raconte.

Autrement dit, **le versant « échec » du chapitre n'est lisible dans aucune
cellule source** : il n'apparaît qu'en bloc, noyé dans le vidage de
`Descent.lean` qu'un carnet imprime. C'est exactement la famille que la section 4
exhibe à la main — et c'est donc ce carnet-ci qui la nomme pour la première fois
dans une cellule source.

Une précaution compte ici autant que le chiffre : **le carnet s'exclut de son
propre balayage**. Il nomme les déclarations qu'il mesure ; s'il se comptait, la
mesure serait vraie par construction et ne dirait rien.


## 7. Exercices

### Exercice 1 — resserrer la barrière `B`

La borne `4 ≤ 721` de la section 3 vient de `B = cost(s₀)`, la barrière la plus
lâche possible. Prenez `B` = le coût du **meilleur** état de l'instance, et
vérifiez que `descent_flips_le_barrier` tient toujours — puis expliquez pourquoi
`run_length_le_cost` ne peut pas, elle, se resserrer sans changer d'hypothèse.


In [7]:
# Exercice 1 : reprendre la barriere de la section 3.
# Indice : `hbarrier` exige que le cout de TOUT etat du run reste sous B.
B_resserre = None   # TODO etudiant : le cout du meilleur etat de l'instance
print("Exercice a completer : resserrer la barriere B et reverifier le fait.")

Exercice a completer : resserrer la barriere B et reverifier le fait.


### Exercice 2 — rendre le plafond `M_N` serré

`descent_target_before_ceiling` conclut `rest.length < M_N` dès que `B < M_N`.
Sur l'instance de la section 3 le run ne fait que 4 flips, très loin de la
borne. Construisez un état de départ dont le run consomme **exactement** `B`
flips — ou montrez que sur cette instance un tel départ n'existe pas.


In [8]:
# Exercice 2 : chercher un depart dont le run epuise la barriere.
# Indice : parcourir les 2^N etats et garder celui dont le run est le plus long.
plus_long = None   # TODO etudiant : (etat de depart, nombre de flips)
print("Exercice a completer : chercher un depart dont le run atteint B flips.")

Exercice a completer : chercher un depart dont le run atteint B flips.


### Exercice 3 — mesurer le seuil au lieu de le dériver

La section 5 **dérive** `t ≥ √(2·log(2N/ε))` du union bound. Mesurez-le : pour
`N` colonnes simulées, estimez par Monte-Carlo la plus petite valeur de `t` telle
que la probabilité qu'**une** colonne sorte de l'intervalle passe sous `ε`, et
comparez au seuil dérivé. L'écart attendu est-il un défaut de la borne, ou son
caractère majorant ?


In [9]:
# Exercice 3 : confronter le seuil derive a un seuil mesure.
# Indice : N colonnes = N vecteurs gaussiens de dimension M, un par experience.
seuil_mesure = None   # TODO etudiant : plus petit t tel que P(une colonne sorte) < eps
print("Exercice a completer : mesurer le seuil par Monte-Carlo et le comparer.")

Exercice a completer : mesurer le seuil par Monte-Carlo et le comparer.


## 8. Conclusion — ce que ce capstone laisse au lecteur

Trois choses, dans l'ordre où elles se sont présentées :

1. **La Proposition 9.1 est vraie et elle est grossière.** Les cinq énoncés de
   `Descent.lean` se vérifient sur une instance réelle, et le plus utile est le
   plus faible : `run_length_le_cost` ne borne les flips que par le coût de
   départ, parce que sa preuve n'utilise que l'intégralité du coût. Resserrer
   demande la géométrie de `Objective.lean`.
2. **Le défaut a un nom, et on peut le produire.** `DescentFailure` n'est pas une
   clause de style : sur un canal identité sans bruit, le témoin s'exhibe à la
   main, et le run revient sur son départ. La stricte décroissance n'est pas une
   commodité de preuve, c'est ce qui borne l'algorithme.
3. **Les instruments aussi se relisent.** Le lake porte `57` déclarations ;
   l'organe canonique en compte `56`, parce que son motif de nom s'arrête au
   premier caractère grec. Le chiffre qui circule dans les rapports est faux
   d'une unité, et il l'est resté parce que personne ne l'avait recompté depuis
   les sources.

Ce que ce carnet **ne fait pas** : il n'exécute pas Lean. Les énoncés formels
sont lus dans les sources, leur vérification native est le travail de
[MIMO-03](MIMO-03-Converse-Native-Lean.ipynb) ; ici, c'est l'**expérience** qui
répond, et la frontière entre les deux est marquée à chaque section.
